In [1]:
conda install psycopg2

Jupyter detected...
2 channel Terms of Service accepted
doneieving notices: - 
Channels:
 - defaults
Platform: osx-arm64
doneecting package metadata (repodata.json): / 
doneing environment: | 


==> WARNING: A newer version of conda exists. <==
    current version: 25.5.1
    latest version: 26.5.0

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /opt/anaconda3

  added / updated specs:
    - psycopg2


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ca-certificates-2026.3.19  |       hca03da5_0         126 KB
    certifi-2026.4.22          |  py313hca03da5_0         132 KB
    openssl-3.5.6              |       ha0b305a_0         4.6 MB
    psycopg2-2.9.11            |  py313h08ee000_0         222 KB
    ------------------------------------------------------------
                                           Total

In [2]:
import psycopg2

функция подключения к базе

In [4]:
def connect_db():
    con = psycopg2.connect(
        database="pythonchik",
        user="postgres",
        password="1111",
        host="127.0.0.1",
        port="5432"
    )

    print("Подключение установлено!")
    return con

создание таблиц 

In [5]:
def create_tables():
    con = connect_db()
    cur = con.cursor()

    # Таблица районов
    cur.execute("""
        CREATE TABLE IF NOT EXISTS districts (
            district_id SERIAL PRIMARY KEY,
            district_name VARCHAR(100) NOT NULL,
            region_name VARCHAR(100) NOT NULL,
            administration_head VARCHAR(100)
        );
    """)

    # Таблица культур
    cur.execute("""
        CREATE TABLE IF NOT EXISTS crops (
            crop_id SERIAL PRIMARY KEY,
            crop_name VARCHAR(100) NOT NULL,
            family_name VARCHAR(100) NOT NULL
        );
    """)

    # Таблица урожайности
    cur.execute("""
        CREATE TABLE IF NOT EXISTS crop_yields (
            yield_id SERIAL PRIMARY KEY,
            district_id INTEGER REFERENCES districts(district_id)
            ON DELETE CASCADE,

            crop_id INTEGER REFERENCES crops(crop_id)
            ON DELETE CASCADE,

            yield_value NUMERIC(10,2) NOT NULL,
            year INTEGER NOT NULL
        );
    """)

    con.commit()
    print("Таблицы успешно созданы!")
    con.close()

ДОБАВЛЕНИЕ ДАННЫХ В districts

In [6]:
def insert_one_district(district_name, region_name, administration_head):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        INSERT INTO districts
        (district_name, region_name, administration_head)

        VALUES (%s, %s, %s)
    """, (district_name, region_name, administration_head))

    con.commit()

    print("Добавлен 1 район")
    print(cur.statusmessage)

    con.close()


def insert_many_districts(data):

    con = connect_db()
    cur = con.cursor()

    cur.executemany("""
        INSERT INTO districts
        (district_name, region_name, administration_head)

        VALUES (%s, %s, %s)
    """, data)

    con.commit()

    print("Добавлено несколько районов")

    con.close()

ДОБАВЛЕНИЕ ДАННЫХ В crops

In [7]:
def insert_one_crop(crop_name, family_name):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        INSERT INTO crops
        (crop_name, family_name)

        VALUES (%s, %s)
    """, (crop_name, family_name))

    con.commit()

    print("Добавлена 1 культура")

    con.close()


def insert_many_crops(data):

    con = connect_db()
    cur = con.cursor()

    cur.executemany("""
        INSERT INTO crops
        (crop_name, family_name)

        VALUES (%s, %s)
    """, data)

    con.commit()

    print("Добавлено несколько культур")

    con.close()



In [8]:
def insert_one_yield(district_id, crop_id, yield_value, year):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        INSERT INTO crop_yields
        (district_id, crop_id, yield_value, year)

        VALUES (%s, %s, %s, %s)
    """, (district_id, crop_id, yield_value, year))

    con.commit()

    print("Добавлена 1 запись урожайности")

    con.close()


def insert_many_yields(data):

    con = connect_db()
    cur = con.cursor()

    cur.executemany("""
        INSERT INTO crop_yields
        (district_id, crop_id, yield_value, year)

        VALUES (%s, %s, %s, %s)
    """, data)

    con.commit()

    print("Добавлено несколько записей урожайности")

    con.close()


In [9]:
def select_all_districts():

    con = connect_db()
    cur = con.cursor()

    cur.execute("SELECT * FROM districts")

    rows = cur.fetchall()

    print("\nТАБЛИЦА districts\n")

    for row in rows:
        print(row)

    con.close()


def select_district_by_region(region_name):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        SELECT * FROM districts
        WHERE region_name = %s
    """, (region_name,))

    rows = cur.fetchall()

    print("\nРайоны по области\n")

    for row in rows:
        print(row)

    con.close()


In [10]:
def select_all_crops():

    con = connect_db()
    cur = con.cursor()

    cur.execute("SELECT * FROM crops")

    rows = cur.fetchall()

    print("\nТАБЛИЦА crops\n")

    for row in rows:
        print(row)

    con.close()


def select_crop_by_family(family_name):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        SELECT * FROM crops
        WHERE family_name = %s
    """, (family_name,))

    rows = cur.fetchall()

    print("\nКультуры по семейству\n")

    for row in rows:
        print(row)

    con.close()


In [11]:
def select_join_data(year):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        SELECT
            d.district_name,
            c.crop_name,
            cy.yield_value,
            cy.year

        FROM crop_yields cy

        JOIN districts d
        ON cy.district_id = d.district_id

        JOIN crops c
        ON cy.crop_id = c.crop_id

        WHERE cy.year = %s
    """, (year,))

    rows = cur.fetchall()

    print("\nДАННЫЕ ПО УРОЖАЙНОСТИ\n")

    for row in rows:
        print(row)

    con.close()


In [12]:
def update_district_head(district_id, new_head):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        UPDATE districts
        SET administration_head = %s
        WHERE district_id = %s
    """, (new_head, district_id))

    con.commit()

    print("Обновлено районов:", cur.rowcount)

    con.close()


def update_crop_family(crop_id, new_family):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        UPDATE crops
        SET family_name = %s
        WHERE crop_id = %s
    """, (new_family, crop_id))

    con.commit()

    print("Обновлено культур:", cur.rowcount)

    con.close()


def update_yield(yield_id, new_value):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        UPDATE crop_yields
        SET yield_value = %s
        WHERE yield_id = %s
    """, (new_value, yield_id))

    con.commit()

    print("Обновлено записей:", cur.rowcount)

    con.close()


In [13]:
def delete_district(district_id):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        DELETE FROM districts
        WHERE district_id = %s
    """, (district_id,))

    con.commit()

    print("Удалено районов:", cur.rowcount)

    con.close()


def delete_crop(crop_id):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        DELETE FROM crops
        WHERE crop_id = %s
    """, (crop_id,))

    con.commit()

    print("Удалено культур:", cur.rowcount)

    con.close()


def delete_yield(yield_id):

    con = connect_db()
    cur = con.cursor()

    cur.execute("""
        DELETE FROM crop_yields
        WHERE yield_id = %s
    """, (yield_id,))

    con.commit()

    print("Удалено записей:", cur.rowcount)

    con.close()


In [14]:
def delete_all_data():

    con = connect_db()
    cur = con.cursor()

    cur.execute("DELETE FROM crop_yields")
    cur.execute("DELETE FROM districts")
    cur.execute("DELETE FROM crops")

    con.commit()

    print("Все данные удалены!")

    con.close()

In [16]:
create_tables()

districts_data = [

    ("Сараевский", "Рязанская", "Толмачев"),
    ("Сасовский", "Рязанская", "Рыбин"),
    ("Пронский", "Рязанская", "Казаков"),
    ("Ейский", "Краснодарский край", "Тимофеев")

]

insert_many_districts(districts_data)

crops_data = [

    ("Пшеница", "Злаки"),
    ("Картофель", "Пасленовые"),
    ("Кукуруза", "Злаки"),
    ("Рожь", "Злаки")

]

insert_many_crops(crops_data)


yield_data = [

    (1, 1, 25.4, 2022),
    (1, 2, 140.0, 2023),
    (2, 1, 30.2, 2024),
    (3, 3, 50.0, 2024),
    (4, 4, 20.5, 2023)

]

insert_many_yields(yield_data)


select_all_districts()

select_district_by_region("Рязанская")

select_all_crops()

select_crop_by_family("Злаки")

select_join_data(2024)


update_district_head(1, "Иванов")

update_crop_family(2, "Овощные")

update_yield(1, 40.7)


delete_yield(5)
delete_district(4)
delete_crop(4)


select_join_data(2024)

Подключение установлено!
Таблицы успешно созданы!
Подключение установлено!
Добавлено несколько районов
Подключение установлено!
Добавлено несколько культур
Подключение установлено!
Добавлено несколько записей урожайности
Подключение установлено!

ТАБЛИЦА districts

(1, 'Сараевский', 'Рязанская', 'Толмачев')
(2, 'Сасовский', 'Рязанская', 'Рыбин')
(3, 'Пронский', 'Рязанская', 'Казаков')
(4, 'Ейский', 'Краснодарский край', 'Тимофеев')
Подключение установлено!

Районы по области

(1, 'Сараевский', 'Рязанская', 'Толмачев')
(2, 'Сасовский', 'Рязанская', 'Рыбин')
(3, 'Пронский', 'Рязанская', 'Казаков')
Подключение установлено!

ТАБЛИЦА crops

(1, 'Пшеница', 'Злаки')
(2, 'Картофель', 'Пасленовые')
(3, 'Кукуруза', 'Злаки')
(4, 'Рожь', 'Злаки')
Подключение установлено!

Культуры по семейству

(1, 'Пшеница', 'Злаки')
(3, 'Кукуруза', 'Злаки')
(4, 'Рожь', 'Злаки')
Подключение установлено!

ДАННЫЕ ПО УРОЖАЙНОСТИ

('Сасовский', 'Пшеница', Decimal('30.20'), 2024)
('Пронский', 'Кукуруза', Decimal('50.0